In [0]:


data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("FlipkartPipeline").getOrCreate()

data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

df_bronze = spark.createDataFrame(data, columns)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Convert types
df_silver = df_bronze.withColumn("amount", col("amount").cast("int")) \
                     .withColumn("date", col("date").cast("date"))

# Handle nulls
df_silver = df_silver.fillna({
    "amount": 0,
    "city": "Unknown"
})

# Remove negative values
df_silver = df_silver.filter(col("amount") >= 0)

# Remove duplicates
df_silver = df_silver.dropDuplicates()

# Keep latest record per order_id
window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())

df_silver = df_silver.withColumn("rn", row_number().over(window_spec)) \
                     .filter(col("rn") == 1) \
                     .drop("rn")

Requirement 1: Sales Analysis
          
        Total sales by product




In [0]:
from pyspark.sql.functions import sum

df_sales_product = df_silver.groupBy("product") \
                           .agg(sum("amount").alias("total_sales")) \
                           .orderBy("total_sales", ascending=False)

df_sales_product.show()

+----------+-----------+
|   product|total_sales|
+----------+-----------+
|    Laptop|     105000|
|    Mobile|      33000|
|    Tablet|      22000|
|     Watch|       8000|
|Headphones|       3000|
+----------+-----------+



Total sales by category



In [0]:
df_sales_category = df_silver.groupBy("category") \
                            .agg(sum("amount").alias("total_sales")) \
                            .orderBy("total_sales", ascending=False)

df_sales_category.show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|     160000|
|Accessories|      11000|
+-----------+-----------+




Total sales by city


In [0]:
df_sales_city = df_silver.groupBy("city") \
                        .agg(sum("amount").alias("total_sales")) \
                        .orderBy("total_sales", ascending=False)

df_sales_city.show()

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|Hyderabad|      72000|
|    Delhi|      55000|
|  Chennai|      33000|
|   Mumbai|       8000|
|  Unknown|       3000|
+---------+-----------+



Requirement 2: Customer Insights
          Number of orders per customer



In [0]:
from pyspark.sql.functions import count

df_orders_customer = df_silver.groupBy("customer_id") \
                             .agg(count("order_id").alias("total_orders")) \
                             .orderBy("total_orders", ascending=False)

df_orders_customer.show()

+-----------+------------+
|customer_id|total_orders|
+-----------+------------+
|       C001|           2|
|       C002|           2|
|       C003|           1|
|       C004|           1|
|       C005|           1|
|       C007|           1|
+-----------+------------+



Top customers based on spending

In [0]:
from pyspark.sql.functions import sum

df_top_customers = df_silver.groupBy("customer_id") \
                           .agg(sum("amount").alias("total_spent")) \
                           .orderBy("total_spent", ascending=False)

df_top_customers.show()

+-----------+-----------+
|customer_id|total_spent|
+-----------+-----------+
|       C001|      72000|
|       C003|      55000|
|       C002|      18000|
|       C007|      15000|
|       C004|       8000|
|       C005|       3000|
+-----------+-----------+



Requirement 3: Data Quality
        No NULL values in critical columns



In [0]:
from pyspark.sql.functions import col

df_null_check = df_silver.filter(
    col("order_id").isNull() |
    col("customer_id").isNull() |
    col("amount").isNull()
)

df_null_check.show()

+--------+-----------+-------+--------+----+----+------+--------+
|order_id|customer_id|product|category|city|date|amount|quantity|
+--------+-----------+-------+--------+----+----+------+--------+
+--------+-----------+-------+--------+----+----+------+--------+



No duplicate orders

In [0]:
from pyspark.sql.functions import count

df_duplicates = df_silver.groupBy("order_id") \
                        .agg(count("*").alias("cnt")) \
                        .filter(col("cnt") > 1)

df_duplicates.show()

+--------+---+
|order_id|cnt|
+--------+---+
+--------+---+




No negative sales


In [0]:
df_negative = df_silver.filter(col("amount") < 0)

df_negative.show()

+--------+-----------+-------+--------+----+----+------+--------+
|order_id|customer_id|product|category|city|date|amount|quantity|
+--------+-----------+-------+--------+----+----+------+--------+
+--------+-----------+-------+--------+----+----+------+--------+




Requirement 4: Latest Data Only
If same order_id appears multiple times → keep latest record


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())

df_latest = df_bronze.withColumn("rn", row_number().over(window_spec)) \
                    .filter(col("rn") == 1) \
                    .drop("rn")

df_latest.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  NULL|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+---

Requirement 5: Dashboard Ready Data
Data should be clean and aggregated
Ready for tools like Power BI


Requirement 5: Dashboard Ready Data
Data should be clean and aggregated
Ready for tools like Power BI


In [0]:
df_silver

DataFrame[order_id: bigint, customer_id: string, product: string, category: string, city: string, date: date, amount: int, quantity: bigint]

In [0]:
from pyspark.sql.functions import sum

df_gold_dashboard = df_silver.groupBy(
    "date", "city", "category", "product"
).agg(
    sum("amount").alias("total_sales"),
    sum("quantity").alias("total_quantity")
)

df_gold_dashboard.show()

+----------+---------+-----------+----------+-----------+--------------+
|      date|     city|   category|   product|total_sales|total_quantity|
+----------+---------+-----------+----------+-----------+--------------+
|2024-01-01|Hyderabad|Electronics|    Laptop|      50000|             1|
|2024-01-02|  Chennai|Electronics|    Mobile|          0|             2|
|2024-01-07|Hyderabad|Electronics|    Tablet|      22000|             1|
|2024-01-04|    Delhi|Electronics|    Laptop|      55000|             1|
|2024-01-05|  Chennai|Electronics|    Mobile|      18000|             1|
|2024-01-06|   Mumbai|Accessories|     Watch|       8000|             1|
|2024-01-08|  Unknown|Accessories|Headphones|       3000|             1|
|2024-01-10|  Chennai|Electronics|    Mobile|      15000|             2|
+----------+---------+-----------+----------+-----------+--------------+



In [0]:
from pyspark.sql.functions import countDistinct

df_gold_dashboard = df_silver.groupBy(
    "date", "city", "category", "product"
).agg(
    sum("amount").alias("total_sales"),
    sum("quantity").alias("total_quantity"),
    countDistinct("customer_id").alias("unique_customers")
)

df_gold_dashboard.show()

+----------+---------+-----------+----------+-----------+--------------+----------------+
|      date|     city|   category|   product|total_sales|total_quantity|unique_customers|
+----------+---------+-----------+----------+-----------+--------------+----------------+
|2024-01-01|Hyderabad|Electronics|    Laptop|      50000|             1|               1|
|2024-01-02|  Chennai|Electronics|    Mobile|          0|             2|               1|
|2024-01-07|Hyderabad|Electronics|    Tablet|      22000|             1|               1|
|2024-01-04|    Delhi|Electronics|    Laptop|      55000|             1|               1|
|2024-01-05|  Chennai|Electronics|    Mobile|      18000|             1|               1|
|2024-01-06|   Mumbai|Accessories|     Watch|       8000|             1|               1|
|2024-01-08|  Unknown|Accessories|Headphones|       3000|             1|               1|
|2024-01-10|  Chennai|Electronics|    Mobile|      15000|             2|               1|
+---------

In [0]:
df_gold_dashboard = df_gold_dashboard.orderBy("date")

In [0]:
df_gold_dashboard.write.mode("overwrite").saveAsTable("gold_dashboard")

In [0]:

spark.sql("SELECT * FROM gold_dashboard").show()

+----------+---------+-----------+----------+-----------+--------------+----------------+
|      date|     city|   category|   product|total_sales|total_quantity|unique_customers|
+----------+---------+-----------+----------+-----------+--------------+----------------+
|2024-01-01|Hyderabad|Electronics|    Laptop|      50000|             1|               1|
|2024-01-02|  Chennai|Electronics|    Mobile|          0|             2|               1|
|2024-01-04|    Delhi|Electronics|    Laptop|      55000|             1|               1|
|2024-01-05|  Chennai|Electronics|    Mobile|      18000|             1|               1|
|2024-01-06|   Mumbai|Accessories|     Watch|       8000|             1|               1|
|2024-01-07|Hyderabad|Electronics|    Tablet|      22000|             1|               1|
|2024-01-08|  Unknown|Accessories|Headphones|       3000|             1|               1|
|2024-01-10|  Chennai|Electronics|    Mobile|      15000|             2|               1|
+---------

3. BRONZE LAYER (What Students Should Do)

Tasks:
Load dataset
Store in Delta format
Use append mode


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("FlipkartPipeline").getOrCreate()

data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

df_bronze = spark.createDataFrame(data, columns)

df_bronze.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  NULL|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|   

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze_orders")


In [0]:
spark.sql("SELECT * FROM bronze_orders").show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  NULL|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|   

4. SILVER LAYER (What Students Should Do)
Data Cleaning
Handle NULL values (amount, city)
Decide: fill or remove



In [0]:
df_bronze = spark.table("bronze_orders")
df_bronze.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  NULL|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|   

In [0]:
from pyspark.sql.functions import col

df_bronze.filter(
    col("amount").isNull() | col("city").isNull()
).show()

+--------+-----------+----------+-----------+-------+----------+------+--------+
|order_id|customer_id|   product|   category|   city|      date|amount|quantity|
+--------+-----------+----------+-----------+-------+----------+------+--------+
|     102|       C002|    Mobile|Electronics|Chennai|2024-01-02|  NULL|       2|
|     107|       C005|Headphones|Accessories|   NULL|2024-01-08|  3000|       1|
|     102|       C002|    Mobile|Electronics|Chennai|2024-01-02|  NULL|       2|
|     107|       C005|Headphones|Accessories|   NULL|2024-01-08|  3000|       1|
+--------+-----------+----------+-----------+-------+----------+------+--------+



In [0]:
df_silver = df_bronze.fillna({
    "amount": 0,
    "city": "Unknown"
})

Data Type Fix
Convert amount → integer
Convert date → proper date format


In [0]:
df_silver = spark.table("silver_orders")

In [0]:
from pyspark.sql.functions import col

df_silver = df_bronze.withColumn("amount", col("amount").cast("int")) \
                     .withColumn("date", col("date").cast("date"))

In [0]:
df_silver.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: date (nullable = true)
 |-- amount: integer (nullable = true)
 |-- quantity: long (nullable = true)



In [0]:
df_silver.filter(col("amount").isNull()).show()

+--------+-----------+-------+-----------+-------+----------+------+--------+
|order_id|customer_id|product|   category|   city|      date|amount|quantity|
+--------+-----------+-------+-----------+-------+----------+------+--------+
|     102|       C002| Mobile|Electronics|Chennai|2024-01-02|  NULL|       2|
|     102|       C002| Mobile|Electronics|Chennai|2024-01-02|  NULL|       2|
+--------+-----------+-------+-----------+-------+----------+------+--------+



In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders")

Remove Duplicates
Identify duplicate order_id
Keep only one record



In [0]:
from pyspark.sql.functions import count, col

df_duplicates = df_silver.groupBy("order_id") \
                        .agg(count("*").alias("cnt")) \
                        .filter(col("cnt") > 1)

df_duplicates.show()

+--------+---+
|order_id|cnt|
+--------+---+
|     102|  2|
|     104|  2|
|     106|  2|
|     109|  4|
|     101|  2|
|     108|  2|
|     107|  2|
|     103|  4|
|     105|  2|
+--------+---+



In [0]:
df_silver = df_silver.dropDuplicates(["order_id"])

In [0]:
df_silver.groupBy("order_id") \
         .count() \
         .filter(col("count") > 1) \
         .show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



Handle Updates
Same order_id appears multiple times



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())

df_silver = df_silver.withColumn("rn", row_number().over(window_spec)) \
                     .filter(col("rn") == 1) \
                     .drop("rn")

In [0]:
df_silver.groupBy("order_id") \
         .count() \
         .filter(col("count") > 1) \
         .show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



Keep latest record based on date
Data Validation



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())

df_silver = df_silver.withColumn("rn", row_number().over(window_spec)) \
                     .filter(col("rn") == 1) \
                     .drop("rn")

In [0]:
df_silver.filter(
    col("order_id").isNull() |
    col("customer_id").isNull() |
    col("amount").isNull()
).show()

+--------+-----------+-------+-----------+-------+----------+------+--------+
|order_id|customer_id|product|   category|   city|      date|amount|quantity|
+--------+-----------+-------+-----------+-------+----------+------+--------+
|     102|       C002| Mobile|Electronics|Chennai|2024-01-02|  NULL|       2|
+--------+-----------+-------+-----------+-------+----------+------+--------+



In [0]:
df_silver.filter(col("amount") < 0).show()

+--------+-----------+-------+-----------+---------+----------+------+--------+
|order_id|customer_id|product|   category|     city|      date|amount|quantity|
+--------+-----------+-------+-----------+---------+----------+------+--------+
|     108|       C006| Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
+--------+-----------+-------+-----------+---------+----------+------+--------+



In [0]:
df_silver.groupBy("order_id") \
         .count() \
         .filter(col("count") > 1) \
         .show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [0]:
print("Null records:", df_silver.filter(
    col("order_id").isNull() |
    col("customer_id").isNull() |
    col("amount").isNull()
).count())

print("Negative records:", df_silver.filter(col("amount") < 0).count())

print("Duplicate records:", df_silver.groupBy("order_id") \
      .count().filter(col("count") > 1).count())

Null records: 1
Negative records: 1
Duplicate records: 0


In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders")


Remove negative amounts
Ensure valid data
Standardization


In [0]:
from pyspark.sql.functions import col

df_silver = df_silver.filter(col("amount") >= 0)

In [0]:
df_silver = df_silver.filter(
    col("order_id").isNotNull() &
    col("customer_id").isNotNull() &
    col("amount").isNotNull()
)

In [0]:
from pyspark.sql.functions import upper, initcap, trim

df_silver = df_silver.withColumn("city", initcap(trim(col("city")))) \
                     .withColumn("product", initcap(trim(col("product")))) \
                     .withColumn("category", initcap(trim(col("category"))))

In [0]:
from pyspark.sql.functions import to_date

df_silver = df_silver.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

In [0]:
df_silver.show()
df_silver.printSchema()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+--------+-----------+----------+-----------+---------+----------+------+--------+

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)


In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders")

5. GOLD LAYER (What Students Should Do)

Create business-level insights

Tasks:
Aggregations
Total sales by product
Total sales by category
Total sales by city


In [0]:
df_silver = spark.table("silver_orders")
df_silver.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+--------+-----------+----------+-----------+---------+----------+------+--------+



In [0]:
from pyspark.sql.functions import sum

df_product_sales = df_silver.groupBy("product") \
                           .agg(sum("amount").alias("total_sales")) \
                           .orderBy("total_sales", ascending=False)

df_product_sales.show()

+----------+-----------+
|   product|total_sales|
+----------+-----------+
|    Laptop|     105000|
|    Mobile|      33000|
|    Tablet|      20000|
|     Watch|       8000|
|Headphones|       3000|
+----------+-----------+



In [0]:
df_category_sales = df_silver.groupBy("category") \
                            .agg(sum("amount").alias("total_sales")) \
                            .orderBy("total_sales", ascending=False)

df_category_sales.show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|     158000|
|Accessories|      11000|
+-----------+-----------+



In [0]:
df_city_sales = df_silver.groupBy("city") \
                        .agg(sum("amount").alias("total_sales")) \
                        .orderBy("total_sales", ascending=False)

df_city_sales.show()

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|Hyderabad|      70000|
|    Delhi|      55000|
|  Chennai|      33000|
|   Mumbai|       8000|
|     NULL|       3000|
+---------+-----------+



In [0]:
df_gold = df_silver.groupBy("product", "category", "city") \
                   .agg(sum("amount").alias("total_sales"))

df_gold.show()

+----------+-----------+---------+-----------+
|   product|   category|     city|total_sales|
+----------+-----------+---------+-----------+
|    Laptop|Electronics|Hyderabad|      50000|
|    Tablet|Electronics|Hyderabad|      20000|
|    Laptop|Electronics|    Delhi|      55000|
|    Mobile|Electronics|  Chennai|      33000|
|     Watch|Accessories|   Mumbai|       8000|
|Headphones|Accessories|     NULL|       3000|
+----------+-----------+---------+-----------+



In [0]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_sales")

In [0]:
spark.sql("SELECT * FROM gold_sales").show()

+----------+-----------+---------+-----------+
|   product|   category|     city|total_sales|
+----------+-----------+---------+-----------+
|    Laptop|Electronics|Hyderabad|      50000|
|    Mobile|Electronics|  Chennai|      33000|
|Headphones|Accessories|     NULL|       3000|
|    Laptop|Electronics|    Delhi|      55000|
|    Tablet|Electronics|Hyderabad|      20000|
|     Watch|Accessories|   Mumbai|       8000|
+----------+-----------+---------+-----------+




Customer Metrics
Total orders per customer
Total spending per customer


In [0]:
df_silver = spark.table("silver_orders")

In [0]:
from pyspark.sql.functions import count

df_orders_customer = df_silver.groupBy("customer_id") \
                             .agg(count("order_id").alias("total_orders")) \
                             .orderBy("total_orders", ascending=False)

df_orders_customer.show()

+-----------+------------+
|customer_id|total_orders|
+-----------+------------+
|       C001|           2|
|       C005|           1|
|       C004|           1|
|       C002|           1|
|       C003|           1|
|       C007|           1|
+-----------+------------+



In [0]:
from pyspark.sql.functions import sum

df_spending_customer = df_silver.groupBy("customer_id") \
                               .agg(sum("amount").alias("total_spent")) \
                               .orderBy("total_spent", ascending=False)

df_spending_customer.show()

+-----------+-----------+
|customer_id|total_spent|
+-----------+-----------+
|       C001|      70000|
|       C003|      55000|
|       C002|      18000|
|       C007|      15000|
|       C004|       8000|
|       C005|       3000|
+-----------+-----------+



In [0]:
from pyspark.sql.functions import count, sum

df_gold_customer = df_silver.groupBy("customer_id") \
                           .agg(
                               count("order_id").alias("total_orders"),
                               sum("amount").alias("total_spent")
                           ) \
                           .orderBy("total_spent", ascending=False)

df_gold_customer.show()

+-----------+------------+-----------+
|customer_id|total_orders|total_spent|
+-----------+------------+-----------+
|       C001|           2|      70000|
|       C003|           1|      55000|
|       C002|           1|      18000|
|       C007|           1|      15000|
|       C004|           1|       8000|
|       C005|           1|       3000|
+-----------+------------+-----------+



In [0]:
df_gold_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_customer_metrics")


Top Analysis
Top selling product
Top customer


In [0]:

df_silver = spark.table("silver_orders")

In [0]:
from pyspark.sql.functions import sum, desc

df_top_product = df_silver.groupBy("product") \
                         .agg(sum("amount").alias("total_sales")) \
                         .orderBy(desc("total_sales"))

df_top_product.show(1)

+-------+-----------+
|product|total_sales|
+-------+-----------+
| Laptop|     105000|
+-------+-----------+
only showing top 1 row


In [0]:
df_top_customer = df_silver.groupBy("customer_id") \
                          .agg(sum("amount").alias("total_spent")) \
                          .orderBy(desc("total_spent"))

df_top_customer.show(1)

+-----------+-----------+
|customer_id|total_spent|
+-----------+-----------+
|       C001|      70000|
+-----------+-----------+
only showing top 1 row
